# 3 — ONNX graph optimizations
Runs in the **Jupyter container on node-serve-model**.

| Level | Effect |
|---|---|
| `ORT_DISABLE_ALL` | No optimization |
| `ORT_ENABLE_BASIC` | Constant folding, redundant node elimination |
| `ORT_ENABLE_EXTENDED` | Layout optimization, operator fusion (Linear+ReLU → single kernel) |

In [ ]:
import time
import numpy as np
import onnxruntime as ort

ONNX_PATH = "model_artifacts/smartqueue_ranker.onnx"

def benchmark_session(session, input_dim=64, num_trials=500, batch_size=32):
    name = session.get_inputs()[0].name
    single = np.random.rand(1, input_dim).astype(np.float32)
    for _ in range(20):
        session.run(None, {name: single})
    latencies = []
    for _ in range(num_trials):
        t0 = time.time()
        session.run(None, {name: single})
        latencies.append(time.time() - t0)
    print(f"p50: {np.percentile(latencies, 50)*1000:.2f} ms")
    print(f"p95: {np.percentile(latencies, 95)*1000:.2f} ms")
    print(f"p99: {np.percentile(latencies, 99)*1000:.2f} ms")
    batch = np.random.rand(batch_size, input_dim).astype(np.float32)
    times = []
    for _ in range(200):
        t0 = time.time()
        session.run(None, {name: batch})
        times.append(time.time() - t0)
    print(f"Throughput (batch={batch_size}): {batch_size * 200 / sum(times):.1f} samples/sec")

In [ ]:
for level_name, level in [
    ("ORT_DISABLE_ALL",    ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
    ("ORT_ENABLE_BASIC",   ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
    ("ORT_ENABLE_EXTENDED",ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
]:
    opts = ort.SessionOptions()
    opts.graph_optimization_level = level
    session = ort.InferenceSession(ONNX_PATH, opts, providers=["CPUExecutionProvider"])
    print(f"\n=== {level_name} ===")
    benchmark_session(session)

In [ ]:
# Save the graph-optimized model for use in the FastAPI service
opts = ort.SessionOptions()
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
opts.optimized_model_filepath = "model_artifacts/smartqueue_ranker_opt.onnx"
ort.InferenceSession(ONNX_PATH, opts, providers=["CPUExecutionProvider"])
print("Saved → model_artifacts/smartqueue_ranker_opt.onnx")